In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# ---------- Paths ----------
def find_base_dir():
    env = os.environ.get("PROJECT_DIR")
    if env and (Path(env) / "data" / "raw").exists():
        return Path(env)
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists():
            return p
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askdirectory(title="Select the project folder (the one containing data/raw)")
        root.destroy()
    except Exception:
        chosen = input("Paste the project folder path: ").strip().strip('"')
    if chosen and (Path(chosen) / "data" / "raw").exists():
        return Path(chosen)
    raise FileNotFoundError("data/raw not found in the selected folder.")

BASE_DIR = find_base_dir()
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

def safe_save(save_fn, filename):
    try:
        save_fn(OUT_DIR / filename)
        print("Saved:", OUT_DIR / filename)
    except OSError as e:
        fallback = Path.home() / "part0_outputs"
        fallback.mkdir(exist_ok=True)
        save_fn(fallback / filename)
        print("Could not write to", OUT_DIR, "-", e)
        print("Saved to", fallback / filename, "instead.")

# ---------- 1. Load raw datasets ----------
districts = pd.read_csv(RAW_DIR / "districts.csv")
age_sex = pd.read_csv(RAW_DIR / "age_sex.csv")
life_table = pd.read_csv(RAW_DIR / "life_table_inputs.csv")
scenarios = pd.read_csv(RAW_DIR / "projection_scenarios.csv")
insured = pd.read_csv(RAW_DIR / "insured_transitions.csv")

# ---------- 2. Data quality log entries ----------
dq_log = []

def log_check(check_name, status, details):
    dq_log.append({"check": check_name, "status": status, "details": details})

log_check("row_counts", "PASS",
          f"districts={len(districts)}, age_sex={len(age_sex)}, life_table={len(life_table)}, "
          f"scenarios={len(scenarios)}, insured={len(insured)}")

log_check("unique_keys", "PASS" if (
    districts["district_id"].nunique() == len(districts)
    and len(age_sex[["district_id", "age_start", "sex"]].drop_duplicates()) == len(age_sex)
    and len(insured[["district_id", "origin", "destination"]].drop_duplicates()) == len(insured)
) else "FAIL", "Primary/composite keys verified across tables")

expected_ages = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85]
age_sex_merged = age_sex.merge(
    districts[["district_id", "region", "population_2020", "population_2026",
               "births_2025", "deaths_2025", "arrivals_2025", "departures_2025", "piped_water_pct"]],
    on="district_id", how="left", validate="many_to_one"
)
log_check("age_sex_coverage_and_join", "PASS" if (
    sorted(age_sex["age_start"].unique().tolist()) == expected_ages
    and sorted(age_sex["sex"].unique().tolist()) == ["F", "M"]
    and age_sex.groupby("district_id").size().eq(36).all()
    and len(age_sex_merged) == len(age_sex)
) else "FAIL", "All 24 districts have 18 age bands x 2 sexes; many-to-one join clean")

male_births_nonzero = int((age_sex.loc[age_sex["sex"] == "M", "births_to_women_2025"] != 0).sum())
log_check("male_births_structural_zero", "PASS" if male_births_nonzero == 0 else "WARN",
          f"Non-zero male births rows = {male_births_nonzero}")

missing_water_ids = districts.loc[districts["piped_water_pct"].isna(), "district_id"].tolist()
log_check("missing_indicators", "INFO",
          f"districts.piped_water_pct missing for {missing_water_ids} (2 rows); "
          f"life_table_inputs interval_years/ax blank only at age_start=85 (open interval)")

valid_ranges = (
    districts["piped_water_pct"].dropna().between(0, 100).all()
    and (age_sex["population_2025"] >= 0).all()
    and (age_sex["baseline_deaths_2025"] >= 0).all()
    and age_sex["survival_5yr"].between(0, 1).all()
    and (life_table["person_years"] > 0).all()
    and (life_table["deaths"] >= 0).all()
    and (insured["exposure_py"] > 0).all()
    and (insured["transition_count"] >= 0).all()
    and (insured["initial_lives"] >= 0).all()
)
log_check("valid_ranges", "PASS" if valid_ranges else "FAIL",
          "All populations, flows, exposures, and survival probabilities within valid bounds")

# ---------- 3. Population & event reconciliation ----------
pop25_by_d = age_sex.groupby("district_id", as_index=False)["population_2025"].sum().rename(
    columns={"population_2025": "pop_2025_from_age_sex"}
)
births25_f_by_d = age_sex[age_sex["sex"] == "F"].groupby("district_id", as_index=False)["births_to_women_2025"].sum().rename(
    columns={"births_to_women_2025": "births_2025_from_age_sex"}
)
base_deaths_by_d = age_sex.groupby("district_id", as_index=False)["baseline_deaths_2025"].sum().rename(
    columns={"baseline_deaths_2025": "baseline_deaths_2025_sum"}
)

reconciled = (
    districts.merge(pop25_by_d, on="district_id")
             .merge(births25_f_by_d, on="district_id")
             .merge(base_deaths_by_d, on="district_id")
)
reconciled["net_migration_2025"] = reconciled["arrivals_2025"] - reconciled["departures_2025"]
reconciled["pop_2026_reconstructed"] = (
    reconciled["pop_2025_from_age_sex"]
    + reconciled["births_2025"]
    - reconciled["deaths_2025"]
    + reconciled["net_migration_2025"]
)
reconciled["diff_pop_2026"] = reconciled["pop_2026_reconstructed"] - reconciled["population_2026"]
reconciled["diff_births_2025"] = reconciled["births_2025_from_age_sex"] - reconciled["births_2025"]
reconciled["deaths_scope_diff_2025"] = reconciled["deaths_2025"] - reconciled["baseline_deaths_2025_sum"]

log_check("balance_reconciliation_2026",
          "PASS" if (reconciled["diff_pop_2026"] == 0).all() and (reconciled["diff_births_2025"] == 0).all() else "WARN",
          f"Total pop_2020={reconciled['population_2020'].sum():,}, "
          f"pop_2025={reconciled['pop_2025_from_age_sex'].sum():,}, "
          f"pop_2026_reported={reconciled['population_2026'].sum():,}, "
          f"pop_2026_reconstructed={reconciled['pop_2026_reconstructed'].sum():,}, "
          f"max_abs_diff={reconciled['diff_pop_2026'].abs().max()}")

log_check("death_scope_difference", "INFO",
          f"District deaths_2025 total={reconciled['deaths_2025'].sum():,} vs "
          f"age_sex baseline_deaths_2025 sum={reconciled['baseline_deaths_2025_sum'].sum():,} "
          f"(diff={reconciled['deaths_scope_diff_2025'].sum():,}; district deaths include deaths among 2025 births)")

dq_df = pd.DataFrame(dq_log)

# Show the log first (full text, no truncation), then save
with pd.option_context("display.max_colwidth", None):
    display(dq_df)

safe_save(lambda p: dq_df.to_csv(p, index=False), "part0_data_quality_log.csv")
safe_save(lambda p: reconciled.to_csv(p, index=False), "part0_reconciled_districts.csv")

BASE_DIR: C:\Users\Lennon\Documents\GitHub\Group-4-Assignment-2-Two-


,check,status,details
0,row_counts,PASS,"districts=24, age_sex=864, life_table=18, scenarios=3, insured=96"
1,unique_keys,PASS,Primary/composite keys verified across tables
2,age_sex_coverage_and_join,PASS,All 24 districts have 18 age bands x 2 sexes; many-to-one join clean
3,male_births_structural_zero,PASS,Non-zero male births rows = 0
4,missing_indicators,INFO,"districts.piped_water_pct missing for ['D04', 'D17'] (2 rows); life_table_inputs interval_years/ax blank only at age_start=85 (open interval)"
5,valid_ranges,PASS,"All populations, flows, exposures, and survival probabilities within valid bounds"
6,balance_reconciliation_2026,PASS,"Total pop_2020=2,402,605, pop_2025=2,636,363, pop_2026_reported=2,658,389, pop_2026_reconstructed=2,658,389, max_abs_diff=0"
7,death_scope_difference,INFO,"District deaths_2025 total=37,507 vs age_sex baseline_deaths_2025 sum=36,886 (diff=621; district deaths include deaths among 2025 births)"


Saved: C:\Users\Lennon\Documents\GitHub\Group-4-Assignment-2-Two-\outputs\part0_data_quality_log.csv
Saved: C:\Users\Lennon\Documents\GitHub\Group-4-Assignment-2-Two-\outputs\part0_reconciled_districts.csv
